In [6]:
# imports
import simpy
import numpy as np
import sys
import pandas as pd
import os
from IPython.display import Image
from tqdm import tqdm
import matplotlib.pyplot as plt
import time



In [10]:

class Queue_example1:

    def __init__(self, lamda, mu, sim_time, warmup):

        self.env = simpy.Environment() # initializing simpy enviroment 
        self.server = simpy.Resource(self.env, capacity=1) # Defining a resource with capacity 1
        self.end_time = sim_time # The time simulation terminate
        self.id_current = 0 # keeping track of cusotmers id
        #an event can one of three: (1. arrival, 2. entering service 3. service completion)
        
        self.print = False
        self.mu = mu # service rate
        self.lamda = lamda # inter-arrival rate
        
        ### L computation
        self.num_cust_durations = np.zeros(500) ## the time duration of each each state (state= number of cusotmers in the system)
        self.last_event_time = 0 # the time of the last event - 
        self.num_cust_sys = 0 # keeping track of number of customers in the system
        self.warmup = warmup

    def run(self):

        self.env.process(self.customer_arrivals()) ## Initializing a process
        self.env.run(until=self.end_time) ## Running the simulaiton until self.end_time

    
    
    #########################################################
    ################# Service block ######################### 
    ######################################################### 
    
    
    def service(self, id_customer):

        arrival_time = self.env.now
        
        with self.server.request() as req:
                       
            yield req
            
            yield self.env.timeout(np.random.exponential(1/self.mu))
            
                
            ###################################
            ### Tracking number of customers ##
            ###################################
            
            ########## Computing the time that elapsed since the last event ###############
            tot_time = self.env.now - self.last_event_time  # keeping track of the last event
            
            ###@@@@@@@@@@@@@############## Accumilating the time that the system was under self.num_cust_sys

            ### $$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$ ###############
            if self.env.now  > self.warmup:
                self.num_cust_durations[self.num_cust_sys] += tot_time # Since the number of customers in the system changes

            ### $$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$ ###############

                
            #we compute how much time the system had this number of customers   
            self.num_cust_sys -= 1 # updating number of cusotmers in the system
            self.last_event_time = self.env.now
            
            ###################################
            ###################################
            ###################################            

    #########################################################
    ################# Arrival block ######################### 
    ######################################################### 
    
    
    def customer_arrivals(self):

        while True:

            yield self.env.timeout(np.random.exponential(1/self.lamda))

            self.id_current += 1
            
            id_customer = self.id_current
            
   
            
            ###################################
            ### Tracking number of customers ##
            ###################################

            ########## Computing the time that elapsed since the last event ###############
            tot_time = self.env.now - self.last_event_time
            
            ### $$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$ ###############
            if self.env.now  > self.warmup:
                self.num_cust_durations[self.num_cust_sys] += tot_time
            
            ### $$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$ ###############
                
            self.num_cust_sys += 1
            self.last_event_time = self.env.now
            
            
            ###################################
            ###################################
            ###################################

            self.env.process(self.service(id_customer))


In [13]:
lamda = 1.5
mu = 2
sim_time = 50000
warmup = 5000

queue_example1 = Queue_example1(lamda, mu, sim_time, warmup)
queue_example1.run()

In [15]:
## Converting time durations to the distribution
L_dist = queue_example1.num_cust_durations/queue_example1.num_cust_durations.sum()
L_dist[:10]

array([0.25876918, 0.19044803, 0.14180194, 0.10939426, 0.07865848,
       0.05741484, 0.04180519, 0.03080146, 0.02357352, 0.01747197])